In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Dense, Flatten, Conv2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array
import cv2
from scipy.ndimage import gaussian_filter

# Load and preprocess MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train[:2000]
y_train = y_train[:2000]

# Expand dims and normalize
x_train = np.expand_dims(x_train, -1) / 255.0
x_test = np.expand_dims(x_test, -1) / 255.0
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# Resize to 32x32 and convert to RGB
def preprocess_images(images):
    resized = np.zeros((images.shape[0], 32, 32, 3))
    for i in range(images.shape[0]):
        img = cv2.resize(images[i], (32, 32))
        img_rgb = np.repeat(img[:, :, np.newaxis], 3, axis=2)
        resized[i] = img_rgb
    return resized

x_train_rgb = preprocess_images(x_train)
x_test_rgb = preprocess_images(x_test)

# Load pretrained ResNet base
resnet = ResNet50(weights=None, include_top=False, input_shape=(32, 32, 3))
x = Flatten()(resnet.output)
x = Dense(128, activation='relu')(x)
output = Dense(10, activation='softmax')(x)
model = Model(inputs=resnet.input, outputs=output)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train_rgb, y_train, batch_size=64, epochs=5, validation_split=0.1)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 168s 4s/step - accuracy: 0.2791 - loss: 2.8157 - val_accuracy: 0.1050 - val_loss: 2.3217
Epoch 2/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 117s 4s/step - accuracy: 0.7368 - loss: 0.8493 - val_accuracy: 0.1050 - val_loss: 2.3492
Epoch 3/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 143s 4s/step - accuracy: 0.8335 - loss: 0.5715 - val_accuracy: 0.1300 - val_loss: 2.5170
Epoch 4/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 140s 4s/step - accuracy: 0.8986 - loss: 0.3686 - val_accuracy: 0.1050 - val_loss: 2.7461
Epoch 5/5
29/29 ━━━━━━━━━━━━━━━━━━━━ 143s 4s/step - accuracy: 0.9259 - loss: 0.2411 - val_accuracy: 0.1000 - val_loss: 2.8458


In [3]:
# FGSM Attack
def FGSM(model, x, y, eps=0.1):
    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        pred = model(x_tensor)
        loss = tf.keras.losses.categorical_crossentropy(y, pred)
    grad = tape.gradient(loss, x_tensor)
    adv = x_tensor + eps * tf.sign(grad)
    return tf.clip_by_value(adv, 0, 1)

# Generate adversarial samples (1000)
adv_samples = np.array([FGSM(model, x_train_rgb[i:i+1], y_train[i:i+1]).numpy()[0] for i in range(1000)])
adv_samples = adv_samples.reshape(-1, 32, 32, 3)

# Noise Injection Functions
def apply_gaussian_noise(images, sigma_min=0.0005, sigma_max=0.005):
    noise_std = np.random.uniform(sigma_min, sigma_max, images.shape)
    return np.clip(images + np.random.normal(0, noise_std), 0, 1)

def apply_motion_blur(images, kernel_size=3):
    blurred = np.zeros_like(images)
    for i in range(images.shape[0]):
        kernel = np.zeros((kernel_size, kernel_size))
        kernel[:, kernel_size // 2] = 1 / kernel_size
        for c in range(3):
            blurred[i, :, :, c] = cv2.filter2D(images[i, :, :, c], -1, kernel)
    return np.clip(blurred, 0, 1)

def apply_glass_blur(images, sigma=0.7):
    return np.clip(gaussian_filter(images, sigma=(0, sigma, sigma, 0)), 0, 1)

def apply_coarse_dropout(images, drop_prob=0.1):
    mask = np.random.rand(*images.shape) > drop_prob
    return images * mask

# Inject all noise in sequence
adv_rgn = apply_gaussian_noise(adv_samples)
adv_smb = apply_motion_blur(adv_rgn)
adv_sgb = apply_glass_blur(adv_smb)
adv_rscd = apply_coarse_dropout(adv_sgb)

# Combine clean + noise-injected adversarial data
x_mixed = np.concatenate([x_train_rgb, adv_rscd])
y_mixed = np.concatenate([y_train, y_train])  # same labels

# Retrain on mixed dataset
model.fit(x_mixed, y_mixed, batch_size=64, epochs=5, validation_split=0.1)


Epoch 1/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 178s 4s/step - accuracy: 0.8696 - loss: 0.4552 - val_accuracy: 0.1033 - val_loss: 3.8442
Epoch 2/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.9568 - loss: 0.1533 - val_accuracy: 0.1067 - val_loss: 4.0475
Epoch 3/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.9717 - loss: 0.0936 - val_accuracy: 0.2067 - val_loss: 2.5641
Epoch 4/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 207s 4s/step - accuracy: 0.9636 - loss: 0.1457 - val_accuracy: 0.4000 - val_loss: 1.9485
Epoch 5/5
43/43 ━━━━━━━━━━━━━━━━━━━━ 198s 4s/step - accuracy: 0.9529 - loss: 0.1508 - val_accuracy: 0.5133 - val_loss: 1.6415


In [4]:
x_test_adv = np.array([FGSM(model, x_test_rgb[i:i+1], y_test[i:i+1]).numpy()[0] for i in range(1000)])
x_test_adv = x_test_adv.reshape(-1, 32, 32, 3)

# Apply noises
test_rgn = apply_gaussian_noise(x_test_adv)
test_smb = apply_motion_blur(test_rgn)
test_sgb = apply_glass_blur(test_smb)
test_rscd = apply_coarse_dropout(test_sgb)

# Evaluate
def evaluate(model, name, data):
    acc = model.evaluate(data, y_test[:data.shape[0]], verbose=0)[1] * 100
    print(f"{name} Accuracy: {acc:.2f}%")

evaluate(model, "Clean Test", x_test_rgb[:1000])
evaluate(model, "Adversarial FGSM", x_test_adv)
evaluate(model, "FGSM + RGN", test_rgn)
evaluate(model, "FGSM + RGN + SMB", test_smb)
evaluate(model, "FGSM + RGN + SMB + SGB", test_sgb)
evaluate(model, "FGSM + RGN + SMB + SGB + RSCD", test_rscd)


Clean Test Accuracy: 71.30%
Adversarial FGSM Accuracy: 8.00%
FGSM + RGN Accuracy: 8.00%
FGSM + RGN + SMB Accuracy: 27.70%
FGSM + RGN + SMB + SGB Accuracy: 34.50%
FGSM + RGN + SMB + SGB + RSCD Accuracy: 34.90%


In [5]:
# --- 1. Train base ResNet model (already trained in your setup) ---
# Reuse your `model` from before (or retrain it if needed)

# --- 2. FGSM Attack Function ---
def FGSM(model, x, y, eps=0.1):
    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        pred = model(x_tensor)
        loss = tf.keras.losses.categorical_crossentropy(y, pred)
    grad = tape.gradient(loss, x_tensor)
    adv = x_tensor + eps * tf.sign(grad)
    return tf.clip_by_value(adv, 0, 1)

# --- 3. Fixed-parameter Noise Functions (same during train + test) ---

def apply_fixed_gaussian_noise(images, sigma=0.003):
    noise = np.random.normal(0, sigma, images.shape)
    return np.clip(images + noise, 0, 1)

def apply_fixed_motion_blur(images, kernel_size=3):
    blurred = np.zeros_like(images)
    kernel = np.zeros((kernel_size, kernel_size))
    kernel[:, kernel_size // 2] = 1 / kernel_size
    for i in range(images.shape[0]):
        for c in range(3):
            blurred[i, :, :, c] = cv2.filter2D(images[i, :, :, c], -1, kernel)
    return np.clip(blurred, 0, 1)

def apply_fixed_glass_blur(images, sigma=0.7):
    return np.clip(gaussian_filter(images, sigma=(0, sigma, sigma, 0)), 0, 1)

def apply_fixed_dropout(images, drop_prob=0.1):
    np.random.seed(42)  # For reproducibility
    mask = np.random.rand(*images.shape) > drop_prob
    return images * mask

# --- 4. Generate Fixed Noise FGSM Training Data ---
# Create 2000 adversarial samples
adv_train = np.array([FGSM(model, x_train_rgb[i:i+1], y_train[i:i+1]).numpy()[0] for i in range(500)])

# Apply same noise sequence to all training adversarial samples
adv_train_rgn = apply_fixed_gaussian_noise(adv_train)
adv_train_smb = apply_fixed_motion_blur(adv_train_rgn)
adv_train_sgb = apply_fixed_glass_blur(adv_train_smb)
adv_train_final = apply_fixed_dropout(adv_train_sgb)

# Combine with clean training data
x_train_mixed = np.concatenate([x_train_rgb, adv_train_final])
y_train_mixed = np.concatenate([y_train, y_train])

# --- 5. Retrain Model on Mixed Data ---
model.fit(x_train_mixed, y_train_mixed, batch_size=64, epochs=5, validation_split=0.1)


Epoch 1/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 148s 4s/step - accuracy: 0.9704 - loss: 0.1224 - val_accuracy: 0.6240 - val_loss: 1.1159
Epoch 2/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 148s 4s/step - accuracy: 0.9897 - loss: 0.0409 - val_accuracy: 0.8240 - val_loss: 0.5275
Epoch 3/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 148s 4s/step - accuracy: 0.9919 - loss: 0.0324 - val_accuracy: 0.7840 - val_loss: 0.7337
Epoch 4/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.9648 - loss: 0.1349 - val_accuracy: 0.8120 - val_loss: 0.7438
Epoch 5/5
36/36 ━━━━━━━━━━━━━━━━━━━━ 202s 4s/step - accuracy: 0.9830 - loss: 0.0482 - val_accuracy: 0.8520 - val_loss: 0.5634


In [6]:
# Generate adversarial samples on test set
test_adv = np.array([FGSM(model, x_test_rgb[i:i+1], y_test[i:i+1]).numpy()[0] for i in range(500)])

# Apply same sequence of fixed noise
test_rgn = apply_fixed_gaussian_noise(test_adv)
test_smb = apply_fixed_motion_blur(test_rgn)
test_sgb = apply_fixed_glass_blur(test_smb)
test_final = apply_fixed_dropout(test_sgb)

# Evaluate
def evaluate(model, name, data):
    acc = model.evaluate(data, y_test[:data.shape[0]], verbose=0)[1] * 100
    print(f"{name} Accuracy: {acc:.2f}%")

evaluate(model, "Clean Test", x_test_rgb[:500])
evaluate(model, "FGSM + Fixed Noise", test_final)


Clean Test Accuracy: 92.40%
FGSM + Fixed Noise Accuracy: 74.40%
